In [1]:
# 04 - Gold: feature engineering para modelado
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import json

# Auto-detección de entorno: Colab o local
IS_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

SILVER = PROJECT_ROOT / 'data/silver'
GOLD = PROJECT_ROOT / 'data/gold'
GOLD.mkdir(parents=True, exist_ok=True)

print('IS_COLAB:', IS_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('SILVER:', SILVER)
print('GOLD:', GOLD)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IS_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2
SILVER: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/silver
GOLD: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/gold


In [2]:
# Cargar datasets Silver
silver_1h_path = SILVER / 'silver_market_sentiment_1h.csv'
silver_1d_path = SILVER / 'silver_market_macro_sentiment_1d.csv'

if not silver_1h_path.exists():
    raise FileNotFoundError(f'No existe: {silver_1h_path}')
if not silver_1d_path.exists():
    raise FileNotFoundError(f'No existe: {silver_1d_path}')

df_1h = pd.read_csv(silver_1h_path)
df_1d = pd.read_csv(silver_1d_path)

print('df_1h:', df_1h.shape)
print('df_1d:', df_1d.shape)

df_1h: (1000, 13)
df_1d: (1000, 13)


In [3]:
# Utilidades de features
def prepare_base(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['timestamp'] = pd.to_datetime(out['timestamp'], utc=True, errors='coerce')
    out = out.dropna(subset=['timestamp']).sort_values('timestamp').drop_duplicates(subset=['timestamp'])
    out = out.reset_index(drop=True)

    num_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
    return out

def add_return_and_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['ret_1'] = out['close'].pct_change(1)
    out['ret_3'] = out['close'].pct_change(3)
    out['ret_6'] = out['close'].pct_change(6)
    out['ret_12'] = out['close'].pct_change(12)
    out['ret_24'] = out['close'].pct_change(24)

    out['log_ret_1'] = np.log(out['close'] / out['close'].shift(1))
    out['volatility_24'] = out['log_ret_1'].rolling(24).std()
    out['volatility_72'] = out['log_ret_1'].rolling(72).std()
    return out

def add_trend_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['sma_12'] = out['close'].rolling(12).mean()
    out['sma_24'] = out['close'].rolling(24).mean()
    out['ema_12'] = out['close'].ewm(span=12, adjust=False).mean()
    out['ema_24'] = out['close'].ewm(span=24, adjust=False).mean()

    out['price_to_sma_24'] = out['close'] / out['sma_24']
    out['ema_ratio_12_24'] = out['ema_12'] / out['ema_24']
    return out

def add_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    delta = out['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    out['rsi_14'] = 100 - (100 / (1 + rs))

    ema_fast = out['close'].ewm(span=12, adjust=False).mean()
    ema_slow = out['close'].ewm(span=26, adjust=False).mean()
    out['macd'] = ema_fast - ema_slow
    out['macd_signal'] = out['macd'].ewm(span=9, adjust=False).mean()
    out['macd_hist'] = out['macd'] - out['macd_signal']
    return out

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['hour'] = out['timestamp'].dt.hour
    out['day_of_week'] = out['timestamp'].dt.dayofweek
    out['is_weekend'] = out['day_of_week'].isin([5, 6]).astype(int)
    return out

In [4]:
# Construir Gold 1h (horizonte 24h)
gold_1h = prepare_base(df_1h)
gold_1h = add_return_and_volatility_features(gold_1h)
gold_1h = add_trend_features(gold_1h)
gold_1h = add_momentum_features(gold_1h)
gold_1h = add_time_features(gold_1h)

# Targets de predicción
gold_1h['target_close_t_plus_24h'] = gold_1h['close'].shift(-24)
gold_1h['target_ret_24h'] = (gold_1h['target_close_t_plus_24h'] / gold_1h['close']) - 1
gold_1h['target_direction_24h'] = (gold_1h['target_ret_24h'] > 0).astype(int)

# Señal de inversión simple basada en retorno esperado
def investment_label(ret):
    if pd.isna(ret):
        return np.nan
    if ret >= 0.02:
        return 'muy_aconsejable'
    if ret >= 0.005:
        return 'aconsejable'
    if ret > -0.005:
        return 'neutral'
    if ret > -0.02:
        return 'riesgoso'
    return 'no_aconsejable'

gold_1h['investment_signal_24h'] = gold_1h['target_ret_24h'].apply(investment_label)

# Limpieza final
gold_1h = gold_1h.replace([np.inf, -np.inf], np.nan)
gold_1h = gold_1h.dropna(subset=['open', 'high', 'low', 'close', 'volume'])
print('gold_1h:', gold_1h.shape)

gold_1h: (1000, 38)


In [5]:
# Construir Gold 1d (horizonte multiperiodo)
gold_1d = prepare_base(df_1d)
gold_1d = add_return_and_volatility_features(gold_1d)
gold_1d = add_trend_features(gold_1d)
gold_1d = add_momentum_features(gold_1d)

# Targets 7d y 30d
gold_1d['target_close_t_plus_7d'] = gold_1d['close'].shift(-7)
gold_1d['target_ret_7d'] = (gold_1d['target_close_t_plus_7d'] / gold_1d['close']) - 1
gold_1d['target_close_t_plus_30d'] = gold_1d['close'].shift(-30)
gold_1d['target_ret_30d'] = (gold_1d['target_close_t_plus_30d'] / gold_1d['close']) - 1

# Señal de inversión de mediano plazo (promedio de horizontes)
gold_1d['target_ret_mix'] = gold_1d[['target_ret_7d', 'target_ret_30d']].mean(axis=1)
gold_1d['investment_signal_mid_term'] = gold_1d['target_ret_mix'].apply(investment_label)

gold_1d = gold_1d.replace([np.inf, -np.inf], np.nan)
gold_1d = gold_1d.dropna(subset=['open', 'high', 'low', 'close'])
print('gold_1d:', gold_1d.shape)

gold_1d: (1000, 37)


In [6]:
# Guardar Gold + metadatos
def save_gold(df: pd.DataFrame, name: str, metadata: dict):
    csv_path = GOLD / f'{name}.csv'
    json_path = GOLD / f'{name}.json'
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f'OK -> {csv_path.name} ({len(df):,} filas)')

ts_now = datetime.now(timezone.utc).isoformat()

meta_gold_1h = {
    'dataset': 'gold_btc_features_1h',
    'created_at_utc': ts_now,
    'rows': int(len(gold_1h)),
    'columns': list(gold_1h.columns),
    'target': ['target_close_t_plus_24h', 'target_ret_24h', 'target_direction_24h'],
}

meta_gold_1d = {
    'dataset': 'gold_btc_features_1d',
    'created_at_utc': ts_now,
    'rows': int(len(gold_1d)),
    'columns': list(gold_1d.columns),
    'target': ['target_ret_7d', 'target_ret_30d', 'target_ret_mix'],
}

save_gold(gold_1h, 'gold_btc_features_1h', meta_gold_1h)
save_gold(gold_1d, 'gold_btc_features_1d', meta_gold_1d)

OK -> gold_btc_features_1h.csv (1,000 filas)
OK -> gold_btc_features_1d.csv (1,000 filas)


In [7]:
# Verificación rápida
gold_files = sorted([p.name for p in GOLD.glob('*')])
print('--- GOLD FILES ---')
for f in gold_files:
    print(f)

display(gold_1h[['timestamp', 'close', 'ret_1', 'rsi_14', 'target_ret_24h', 'investment_signal_24h']].tail(5))
display(gold_1d[['timestamp', 'close', 'target_ret_7d', 'target_ret_30d', 'investment_signal_mid_term']].tail(5))

--- GOLD FILES ---
.gitkeep
gold_btc_features_1d.csv
gold_btc_features_1d.json
gold_btc_features_1h.csv
gold_btc_features_1h.json


,timestamp,close,ret_1,rsi_14,target_ret_24h,investment_signal_24h
995,2026-03-11 01:00:00+00:00,70082.03,0.000803,44.727846,NaN,NaN
996,2026-03-11 02:00:00+00:00,69803.80,-0.003970,43.239046,NaN,NaN
997,2026-03-11 03:00:00+00:00,69553.72,-0.003583,46.080580,NaN,NaN
998,2026-03-11 04:00:00+00:00,70150.45,0.008579,33.762473,NaN,NaN
999,2026-03-11 05:00:00+00:00,70116.29,-0.000487,33.862004,NaN,NaN


,timestamp,close,target_ret_7d,target_ret_30d,investment_signal_mid_term
995,2026-03-07 00:00:00+00:00,67264.19,NaN,NaN,NaN
996,2026-03-08 00:00:00+00:00,66007.38,NaN,NaN,NaN
997,2026-03-09 00:00:00+00:00,68374.92,NaN,NaN,NaN
998,2026-03-10 00:00:00+00:00,69937.15,NaN,NaN,NaN
999,2026-03-11 00:00:00+00:00,70116.29,NaN,NaN,NaN
